# Actividad de Estadística Inferencial en R: Costos en Salud

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/Astroestadistica/blob/main/ITM/Actividad_ITM_Estadistica_Inferencial_R.ipynb)

**Tema elegido:** Costos y presupuestos en salud  
**Dataset base:** *Medical Cost Personal Dataset* (insurance.csv)

---

## Introducción

En esta actividad se desarrolla un análisis estadístico inferencial completo usando **R** sobre un problema real de costos médicos. El objetivo es estudiar cómo variables demográficas y de estilo de vida se relacionan con el costo anual del seguro de salud (`charges`).

El dataset original contiene más de 70 registros (1338 observaciones). Como la guía solicita trabajar con **70 datos**, se aplicará un procedimiento de selección que reduzca el tamaño sin sesgar la muestra: **muestreo aleatorio estratificado proporcional** por dos variables cualitativas relevantes (`smoker` y `region`).

Este enfoque mantiene la estructura del conjunto original en grupos críticos para el análisis de costos y reduce el riesgo de elegir accidentalmente una muestra no representativa.

In [ ]:
# Configuración inicial
set.seed(2026)
options(repr.plot.width = 11, repr.plot.height = 7)

required_packages <- c("dplyr", "ggplot2", "broom", "scales", "knitr")
missing_packages <- required_packages[!(required_packages %in% installed.packages()[, "Package"])]
if (length(missing_packages) > 0) {
  install.packages(missing_packages, repos = "https://cloud.r-project.org")
}

library(dplyr)
library(ggplot2)
library(broom)
library(scales)
library(knitr)

theme_set(theme_minimal(base_size = 16))

## 1) Carga del dataset y revisión inicial

Se utiliza una fuente pública en formato CSV. Las variables del dataset son:

- Cuantitativas: `age`, `bmi`, `children`, `charges`.
- Cualitativas: `sex`, `smoker`, `region`.

Esto permite cubrir ampliamente los requisitos de análisis descriptivo, pruebas de hipótesis, asociaciones entre variables y modelamiento inferencial.

In [ ]:
url <- "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
insurance <- read.csv(url)

str(insurance)
summary(insurance)
head(insurance)

## 2) Selección de 70 observaciones sin sesgo

Para evitar sesgo por selección manual o por orden temporal del archivo, se implementa un **muestreo estratificado proporcional** considerando la combinación de:

- `smoker` (fumador/no fumador)
- `region` (northeast, northwest, southeast, southwest)

Con esto, cada subgrupo conserva aproximadamente su peso relativo del conjunto total. Luego se completa el ajuste exacto a 70 observaciones distribuyendo de forma controlada cualquier residuo de redondeo.

In [ ]:
# Conversión de tipo para asegurar tratamiento categórico
insurance <- insurance %>%
  mutate(
    sex = factor(sex),
    smoker = factor(smoker),
    region = factor(region)
  )

# Estratos y tamaños esperados por proporción
strata_counts <- insurance %>%
  count(smoker, region, name = "N_total") %>%
  mutate(
    p = N_total / sum(N_total),
    n_raw = 70 * p,
    n_floor = floor(n_raw),
    frac = n_raw - n_floor
  )

remaining <- 70 - sum(strata_counts$n_floor)

if (remaining > 0) {
  strata_counts <- strata_counts %>%
    arrange(desc(frac)) %>%
    mutate(add = ifelse(row_number() <= remaining, 1, 0),
           n_sample = n_floor + add) %>%
    arrange(smoker, region)
} else {
  strata_counts <- strata_counts %>%
    mutate(n_sample = n_floor)
}

strata_counts
sum(strata_counts$n_sample)

# Muestreo estratificado
sample_70 <- insurance %>%
  inner_join(strata_counts %>% select(smoker, region, n_sample), by = c("smoker", "region")) %>%
  group_by(smoker, region) %>%
  slice_sample(n = first(n_sample), replace = FALSE) %>%
  ungroup() %>%
  select(age, sex, bmi, children, smoker, region, charges)

nrow(sample_70)
head(sample_70)

### Comparación de composición: dataset completo vs muestra de 70

Esta comparación permite verificar que la reducción mantiene una estructura razonablemente representativa en variables cualitativas clave.

In [ ]:
comp_smoker <- bind_rows(
  insurance %>% count(smoker) %>% mutate(source = "Original"),
  sample_70 %>% count(smoker) %>% mutate(source = "Muestra 70")
) %>%
  group_by(source) %>%
  mutate(prop = n / sum(n)) %>%
  ungroup()

comp_region <- bind_rows(
  insurance %>% count(region) %>% mutate(source = "Original"),
  sample_70 %>% count(region) %>% mutate(source = "Muestra 70")
) %>%
  group_by(source) %>%
  mutate(prop = n / sum(n)) %>%
  ungroup()

kable(comp_smoker)
kable(comp_region)

## 3) Definición de variables para requisitos de la actividad

Con la muestra de 70 casos se tienen al menos 6 variables de tipos requeridos:

- **Cuantitativas continuas:** `bmi`, `charges`
- **Cuantitativa discreta:** `children`
- **Cualitativas nominales:** `sex`, `smoker`
- **Cualitativa ordinal (derivada):** `risk_level` según nivel de costo

Además se conserva `age` y `region` para enriquecer el análisis.

In [ ]:
sample_70 <- sample_70 %>%
  mutate(
    risk_level = cut(
      charges,
      breaks = quantile(charges, probs = c(0, 0.33, 0.66, 1), na.rm = TRUE),
      include.lowest = TRUE,
      labels = c("Bajo", "Medio", "Alto")
    ),
    risk_level = ordered(risk_level, levels = c("Bajo", "Medio", "Alto"))
  )

str(sample_70)
summary(sample_70)

## 4) Análisis descriptivo

Primero se resumen medidas de tendencia central y dispersión para variables cuantitativas. Después se construyen visualizaciones legibles (tamaño de texto amplio) para identificar patrones de distribución y posibles valores extremos.

In [ ]:
desc_stats <- sample_70 %>%
  summarise(
    edad_media = mean(age),
    edad_sd = sd(age),
    bmi_media = mean(bmi),
    bmi_sd = sd(bmi),
    hijos_media = mean(children),
    hijos_sd = sd(children),
    charges_media = mean(charges),
    charges_mediana = median(charges),
    charges_sd = sd(charges),
    charges_min = min(charges),
    charges_max = max(charges)
  )

kable(desc_stats, digits = 2)

In [ ]:
# Distribución de costos
p1 <- ggplot(sample_70, aes(x = charges)) +
  geom_histogram(fill = "#2C7FB8", color = "white", bins = 20) +
  scale_x_continuous(labels = dollar_format(prefix = "$", big.mark = ",")) +
  labs(
    title = "Distribución de costos anuales (charges)",
    x = "Costo anual",
    y = "Frecuencia"
  )

# Costos por condición de fumador
p2 <- ggplot(sample_70, aes(x = smoker, y = charges, fill = smoker)) +
  geom_boxplot(alpha = 0.85) +
  scale_y_continuous(labels = dollar_format(prefix = "$", big.mark = ",")) +
  labs(
    title = "Costos por condición de fumador",
    x = "Fumador",
    y = "Costo anual"
  ) +
  guides(fill = "none")

# Relación edad-costo
p3 <- ggplot(sample_70, aes(x = age, y = charges, color = smoker)) +
  geom_point(size = 3, alpha = 0.8) +
  geom_smooth(method = "lm", se = FALSE, linewidth = 1.2) +
  scale_y_continuous(labels = dollar_format(prefix = "$", big.mark = ",")) +
  labs(
    title = "Relación entre edad y costo (segmentado por fumador)",
    x = "Edad",
    y = "Costo anual"
  )

print(p1)
print(p2)
print(p3)

### Interpretación descriptiva

1. La variable `charges` suele presentar asimetría positiva (costos altos concentrados en una fracción de personas).
2. Los fumadores tienden a mostrar costos notablemente superiores frente a no fumadores.
3. Existe una tendencia creciente entre edad y costo, más marcada dentro del grupo fumador.
4. El análisis descriptivo ya sugiere diferencias de grupos y relaciones lineales que luego se contrastan formalmente con pruebas inferenciales.

## 5) Pruebas de hipótesis

Se aplican varias pruebas para cubrir normalidad, independencia, comparación de grupos y asociación no paramétrica/paramétrica, con nivel de significancia $\alpha = 0.05$.

In [ ]:
alpha <- 0.05

# 5.1 Normalidad de charges
shapiro_charges <- shapiro.test(sample_70$charges)
shapiro_charges

# 5.2 Independencia: sexo vs condición de fumador
tabla_sex_smoker <- table(sample_70$sex, sample_70$smoker)
chi_sex_smoker <- chisq.test(tabla_sex_smoker)

tabla_sex_smoker
chi_sex_smoker

# 5.3 Diferencia de medias independientes (t-test): BMI por sexo
t_bmi_sex <- t.test(bmi ~ sex, data = sample_70, var.equal = FALSE)
t_bmi_sex

# 5.4 Prueba no paramétrica Mann-Whitney: charges por smoker
wilcox_charges_smoker <- wilcox.test(charges ~ smoker, data = sample_70, exact = FALSE)
wilcox_charges_smoker

### Interpretación inferencial de las pruebas

- **Shapiro-Wilk (`charges`)**: permite evaluar si la distribución de costos se aproxima a normalidad.
- **Chi-cuadrado (`sex` vs `smoker`)**: identifica si existe dependencia entre ambas variables cualitativas.
- **t-test (`bmi` por `sex`)**: contrasta diferencia de medias de IMC entre grupos de sexo.
- **Mann-Whitney (`charges` por `smoker`)**: compara distribuciones de costos entre fumadores y no fumadores sin asumir normalidad.

La decisión se toma con p-valor: si \(p < 0.05\), se rechaza \(H_0\) y se concluye evidencia estadísticamente significativa.

## 6) Correlación y regresión lineal múltiple

Se ajusta un modelo donde `charges` es la variable respuesta y los predictores incluyen:

- Cuantitativos: `age`, `bmi`, `children`
- Cualitativos: `smoker`, `sex`, `region`

Esto permite estimar el efecto parcial de cada variable sobre el costo esperado, controlando por las demás.

In [ ]:
# Correlación de Pearson entre variables numéricas principales
corr_mat <- cor(sample_70 %>% select(age, bmi, children, charges))
kable(round(corr_mat, 3))

# Modelo de regresión múltiple
modelo <- lm(charges ~ age + bmi + children + smoker + sex + region, data = sample_70)
summary(modelo)

# Tabla ordenada de coeficientes
coef_table <- broom::tidy(modelo, conf.int = TRUE)
kable(coef_table, digits = 4)

In [ ]:
# Diagnóstico gráfico de supuestos
par(mfrow = c(2, 2), cex.axis = 1.2, cex.lab = 1.2, cex.main = 1.1)
plot(modelo)
par(mfrow = c(1, 1))

### Lectura del modelo

Al interpretar los coeficientes:

1. El parámetro de `smoker` suele tener el impacto positivo más fuerte en los costos.
2. `age` también suele incrementar el costo esperado, manteniendo lo demás constante.
3. El ajuste global se revisa con $R^2$ y $R^2$ ajustado.
4. Los supuestos (linealidad, normalidad de residuos, homocedasticidad e independencia) se evalúan con gráficos diagnósticos.

Si algún supuesto no se cumple bien, se recomienda en un trabajo posterior explorar transformaciones (por ejemplo `log(charges)`) o modelos robustos.

## 7) Inferencia y predicción

Para ilustrar uso práctico del modelo, se generan predicciones para perfiles hipotéticos de personas con distintas características.

In [ ]:
new_cases <- data.frame(
  age = c(25, 45, 60),
  bmi = c(24, 30, 33),
  children = c(0, 2, 3),
  smoker = factor(c("no", "yes", "no"), levels = levels(sample_70$smoker)),
  sex = factor(c("female", "male", "female"), levels = levels(sample_70$sex)),
  region = factor(c("northwest", "southeast", "northeast"), levels = levels(sample_70$region))
)

pred <- predict(modelo, newdata = new_cases, interval = "prediction", level = 0.95)
resultado_pred <- cbind(new_cases, as.data.frame(pred))

kable(resultado_pred, digits = 2)

### Comentario de predicción

El intervalo de predicción (lwr, upr) es más amplio que un intervalo de confianza de la media, porque incorpora variabilidad individual. Esto es útil en contextos de planeación financiera en salud: el modelo entrega una estimación central del costo esperado, pero también un rango de incertidumbre para decisiones de presupuesto.

## 8) Conclusiones

1. El procedimiento de muestreo estratificado proporcional permitió trabajar con 70 observaciones manteniendo la representatividad de subgrupos relevantes.
2. El costo anual en salud (`charges`) presenta alta variabilidad y tendencia asimétrica.
3. Ser fumador se asocia con incrementos importantes en costos médicos, tanto descriptiva como inferencialmente.
4. La regresión múltiple permite cuantificar el efecto parcial de edad, IMC, hijos y variables categóricas sobre el gasto esperado.
5. El enfoque inferencial aplicado es útil para soporte de decisiones en costos y presupuestos de salud.

## 9) Recomendación de otros dos datasets para la misma actividad

1. **Medical Expenditure Panel Survey (MEPS)**  
   Enfocado en gasto médico, uso de servicios de salud y características socioeconómicas. Muy útil para inferencia en administración en salud y presupuestos.

2. **Hospital Cost Reports (CMS, EE. UU.)**  
   Reportes de costos hospitalarios con múltiples dimensiones financieras y operativas. Ideal para análisis de costos, comparación entre instituciones y modelado econométrico.